# Notebook 4 — Modelado, Optimización (Optuna) y Pipeline de Reentrenamiento (Walk-Forward)

**Objetivo**: entrenar y comparar dos familias de modelos de goles (Poisson GLM vs
gradient boosting), afinar el modelo de árboles con Optuna, y simular el propio
torneo fase a fase con un **reentrenamiento continuo (rolling update)**: cada ronda
eliminatoria se evalúa con el modelo tal como estaba ANTES de conocer sus resultados,
y solo después esos resultados se incorporan para la ronda siguiente — la disciplina
de validación que evita hacer trampa sin darse cuenta.

Entrada: `data/processed/partidos_features.csv` (Notebook 2). Conclusiones que se
reutilizan del Notebook 3: corte temporal en 1990, y el conjunto de features libre de
colinealidad severa.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import joblib
import json
import lightgbm as lgb
import networkx as nx
import numpy as np
import optuna
import pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import log_loss, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)  # el notebook ya imprime su propio resumen

DIR_RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_PROCESSED = DIR_RAIZ / "data" / "processed"
DIR_MODELOS = DIR_RAIZ / "models"

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

SEMILLA = 42
# Rejilla 0..8: ahora la matriz decide SIEMPRE quién avanza (4.6), incluidos
# partidos con ventaja grande (lambda ~4), no solo cruces parejos -- con 0..5
# se perdía masa de probabilidad real en esos casos.
MAX_GOLES_MATRIZ = 8

# Conclusiones del Notebook 3 (obtenidas allí con evidencia; aquí se consumen
# como constantes documentadas, no se repite el análisis).
FECHA_CORTE_HISTORICO = pd.Timestamp("1990-01-01")
FEATURES_MODELO = [
    "elo_diff", "tendencia_elo_local", "tendencia_elo_visitante",
    "dif_forma_gf_5", "dif_forma_gf_10", "dif_racha_5", "dif_racha_10",
    "dias_descanso_local", "dias_descanso_visitante",
]

df = pd.read_csv(DIR_PROCESSED / "partidos_features.csv", parse_dates=["fecha"])
df = df[df["fecha"] >= FECHA_CORTE_HISTORICO].reset_index(drop=True)
print(f"Partidos desde {FECHA_CORTE_HISTORICO.date()}: {len(df):,}")

## 4.1 Detectar fase de grupos vs. eliminatorias del Mundial 2026

`results.csv` no trae una columna de "ronda" explícita para el Mundial en curso. La
fase de grupos SÍ se puede reconstruir sin ambigüedad: dentro de las selecciones que
han jugado el Mundial 2026, un grupo real es un conjunto de 4 equipos que se han
enfrentado entre sí TODOS contra TODOS (round-robin completo = camarilla de 4 en el
grafo de "quién jugó contra quién"). Un cruce de eliminatoria aislado entre dos
selecciones de grupos distintos no basta para simular una camarilla falsa, así que el
método es robusto aunque ya se hayan jugado partidos de eliminatoria — no hace falta
ninguna fuente externa ni fecha de corte arbitraria para separar ambas fases.

In [2]:
def detectar_fase_grupos(df_mundial: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Separa los partidos del Mundial 2026 en (fase_de_grupos, eliminatorias).

    Construye el grafo de enfrentamientos y busca camarillas de 4 (grupos
    reales); cualquier partido que no sea interno a una de esas camarillas
    es, por descarte, de eliminatoria directa.
    '''
    grafo = nx.Graph()
    for _, fila in df_mundial.iterrows():
        grafo.add_edge(fila["equipo_local"], fila["equipo_visitante"])

    camarillas_de_4 = [c for c in nx.enumerate_all_cliques(grafo) if len(c) == 4]
    usados: set[str] = set()
    grupos: list[frozenset[str]] = []
    for c in camarillas_de_4:
        if not (set(c) & usados):
            grupos.append(frozenset(c))
            usados.update(c)

    def _mismo_grupo(fila) -> bool:
        par = {fila["equipo_local"], fila["equipo_visitante"]}
        return any(par <= g for g in grupos)

    es_fase_grupos = df_mundial.apply(_mismo_grupo, axis=1)
    print(f"Grupos detectados: {len(grupos)} ({sum(len(g) for g in grupos)} selecciones)")
    print(f"Partidos de fase de grupos: {es_fase_grupos.sum()} | de eliminatoria: {(~es_fase_grupos).sum()}")
    return df_mundial[es_fase_grupos].copy(), df_mundial[~es_fase_grupos].copy()


df_mundial_2026 = df[(df["torneo"] == "FIFA World Cup") & (df["fecha"].dt.year == 2026)]
df_grupos, df_eliminatoria = detectar_fase_grupos(df_mundial_2026)

Grupos detectados: 12 (48 selecciones)
Partidos de fase de grupos: 72 | de eliminatoria: 22


## 4.2 Split temporal

- **Entrenamiento**: todo el histórico (desde 1990) hasta el **10 de junio de 2026**,
  el día antes de que arrancara el Mundial — el modelo no ve ni un solo partido del
  torneo antes de predecirlo.
- **Test inicial**: los partidos de **fase de grupos ya jugados**.

In [3]:
FECHA_INICIO_MUNDIAL = pd.Timestamp("2026-06-11")

df_train_inicial = df[(df["fecha"] < FECHA_INICIO_MUNDIAL) & df["jugado"]].copy()
df_test_grupos = df_grupos[df_grupos["jugado"]].copy()

print(f"Entrenamiento inicial: {len(df_train_inicial):,} partidos (hasta {FECHA_INICIO_MUNDIAL.date()})")
print(f"Test (fase de grupos ya jugada): {len(df_test_grupos):,} partidos")

assert df_train_inicial["fecha"].max() < FECHA_INICIO_MUNDIAL, "Fuga temporal: hay partidos del Mundial en el train" 

Entrenamiento inicial: 32,286 partidos (hasta 2026-06-11)
Test (fase de grupos ya jugada): 72 partidos


## 4.3 Modelos base de goles: Poisson GLM vs. Gradient Boosting

Se entrena un modelo INDEPENDIENTE para los goles del local y otro para los del
visitante — mezclar ambos en un único target no tiene sentido (son dos procesos con
features "local"/"visitante" invertidas). Dos familias candidatas:

- **Modelo A — Regresión de Poisson (GLM)**: el fútbol es el caso de libro de un
  proceso de conteo (goles en un tiempo fijo, sucesos raros e independientes entre
  sí); un GLM de Poisson lo modela de forma nativa y sus predicciones son siempre
  tasas positivas, listas para la matriz de probabilidad conjunta de más abajo.
- **Modelo B — LightGBM con objetivo Poisson**: captura relaciones no lineales e
  interacciones entre features que un GLM lineal no puede, al coste de ser una caja
  más negra y de necesitar más datos para no sobreajustar.

In [4]:
def entrenar_poisson_glm(X: pd.DataFrame, y: pd.Series, alpha: float = 0.5):
    '''Regresión de Poisson (GLM) — alpha es la regularización L2; un valor
    demasiado alto aplana las predicciones hacia la media global (síntoma
    clásico: el modelo predice casi el mismo marcador para cualquier partido).

    Escalado obligatorio: `elo_diff` se mueve en el orden de los cientos
    mientras que `dif_racha_5` lo hace en unidades — sin normalizar, el
    solver del GLM queda mal condicionado (produce advertencias de overflow
    en el gradiente) y los coeficientes dejan de ser comparables entre sí.
    '''
    return make_pipeline(StandardScaler(), PoissonRegressor(alpha=alpha, max_iter=1000)).fit(X, y)


def entrenar_lightgbm_poisson(X: pd.DataFrame, y: pd.Series, params: dict | None = None) -> lgb.LGBMRegressor:
    '''LightGBM con objetivo Poisson (tasas de gol, nunca negativas) en vez de
    error cuadrático — con error cuadrático la varianza real del fútbol (mayor
    cuantos más goles se esperan) queda mal representada.
    '''
    params_finales = {"objective": "poisson", "random_state": SEMILLA, "verbosity": -1, "n_estimators": 300}
    params_finales.update(params or {})
    return lgb.LGBMRegressor(**params_finales).fit(X, y)


X_train_inicial = df_train_inicial[FEATURES_MODELO]
X_test_grupos = df_test_grupos[FEATURES_MODELO]

modelos_glm = {
    "local": entrenar_poisson_glm(X_train_inicial, df_train_inicial["goles_local"]),
    "visitante": entrenar_poisson_glm(X_train_inicial, df_train_inicial["goles_visitante"]),
}
modelos_lgbm_base = {
    "local": entrenar_lightgbm_poisson(X_train_inicial, df_train_inicial["goles_local"]),
    "visitante": entrenar_lightgbm_poisson(X_train_inicial, df_train_inicial["goles_visitante"]),
}
print("Modelos base entrenados (Poisson GLM y LightGBM sin ajustar).")

Modelos base entrenados (Poisson GLM y LightGBM sin ajustar).


## 4.4 Optimización de hiperparámetros con Optuna (LightGBM)

In [5]:
def objetivo_optuna(trial: optuna.Trial, X: pd.DataFrame, y: pd.Series, n_splits: int = 5) -> float:
    '''Función objetivo: RMSE medio en validación cruzada TEMPORAL (nunca un
    K-Fold barajado — en series temporales eso entrenaría con 2025 para
    predecir 2010, inflando la métrica de forma escandalosa).
    '''
    params = {
        "objective": "poisson",
        "random_state": SEMILLA,
        "verbosity": -1,
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "num_leaves": trial.suggest_int("num_leaves", 8, 128),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }

    cv = TimeSeriesSplit(n_splits=n_splits)
    rmses = []
    for idx_train, idx_val in cv.split(X):
        modelo = lgb.LGBMRegressor(**params).fit(X.iloc[idx_train], y.iloc[idx_train])
        pred = np.clip(modelo.predict(X.iloc[idx_val]), 0, None)
        rmses.append(np.sqrt(mean_squared_error(y.iloc[idx_val], pred)))
    return float(np.mean(rmses))


def optimizar_lightgbm(X: pd.DataFrame, y: pd.Series, n_trials: int = 40, nombre: str = "") -> optuna.Study:
    estudio = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEMILLA))
    estudio.optimize(lambda trial: objetivo_optuna(trial, X, y), n_trials=n_trials, show_progress_bar=False)
    print(f"[{nombre}] Mejor RMSE (CV temporal): {estudio.best_value:.4f}")
    print(f"[{nombre}] Mejores hiperparámetros: {estudio.best_params}")
    return estudio


estudio_local = optimizar_lightgbm(X_train_inicial, df_train_inicial["goles_local"], nombre="goles_local")
estudio_visitante = optimizar_lightgbm(X_train_inicial, df_train_inicial["goles_visitante"], nombre="goles_visitante")

modelos_lgbm_optuna = {
    "local": entrenar_lightgbm_poisson(X_train_inicial, df_train_inicial["goles_local"], estudio_local.best_params),
    "visitante": entrenar_lightgbm_poisson(X_train_inicial, df_train_inicial["goles_visitante"], estudio_visitante.best_params),
}

[goles_local] Mejor RMSE (CV temporal): 1.5933
[goles_local] Mejores hiperparámetros: {'n_estimators': 452, 'num_leaves': 21, 'learning_rate': 0.016327699589195513, 'min_child_samples': 81, 'reg_lambda': 0.008005788369403466, 'subsample': 0.6006978563343439, 'colsample_bytree': 0.7373415154810438}


[goles_visitante] Mejor RMSE (CV temporal): 1.2673
[goles_visitante] Mejores hiperparámetros: {'n_estimators': 368, 'num_leaves': 8, 'learning_rate': 0.0307109821135174, 'min_child_samples': 38, 'reg_lambda': 0.07841058277510576, 'subsample': 0.6284090669902946, 'colsample_bytree': 0.9402468530565209}


## 4.5 Evaluación en la fase de grupos: GLM vs. LightGBM afinado

Dos métricas, dos preguntas distintas: **RMSE** sobre los goles exactos, y **LogLoss**
sobre el resultado 1X2 derivado de la rejilla conjunta de Poisson (más abajo) — un
modelo puede acertar el marcador y aun así dar probabilidades mal calibradas, o
viceversa.

In [ ]:
def log_verosimilitud_dixon_coles(rho: float, lam: np.ndarray, mu: np.ndarray, x: np.ndarray, y: np.ndarray) -> float:
    '''Log-verosimilitud de los marcadores observados (x, y) bajo la corrección
    de Dixon-Coles, con las lambdas (lam, mu) ya estimadas por el GLM -- se usa
    solo para calibrar rho por rejilla, con las tasas de gol ya fijas (el
    procedimiento en dos pasos del propio paper de 1997).'''
    tau = np.ones_like(lam)
    es_0_0 = (x == 0) & (y == 0)
    es_0_1 = (x == 0) & (y == 1)
    es_1_0 = (x == 1) & (y == 0)
    es_1_1 = (x == 1) & (y == 1)
    tau[es_0_0] = 1 - lam[es_0_0] * mu[es_0_0] * rho
    tau[es_0_1] = 1 + lam[es_0_1] * rho
    tau[es_1_0] = 1 + mu[es_1_0] * rho
    tau[es_1_1] = 1 - rho
    tau = np.clip(tau, 1e-8, None)  # evita log(negativo) si rho se saliera de su rango válido
    return float((np.log(tau) + poisson.logpmf(x, lam) + poisson.logpmf(y, mu)).sum())


def ajustar_rho_dixon_coles(modelos: dict, X: pd.DataFrame, goles_local: pd.Series, goles_visitante: pd.Series) -> float:
    '''Busca por rejilla fina (rho en [-0.3, 0.3], paso 0.005) el valor que
    maximiza la log-verosimilitud de los marcadores realmente observados. Un
    Poisson bivariante independiente infravalora sistemáticamente los
    marcadores bajos (0-0, 1-0, 0-1, 1-1) frente a la correlación negativa
    leve que de verdad existe en el fútbol -- Dixon & Coles (1997) corrigen
    justo esas 4 celdas con este factor.
    '''
    lam = np.clip(modelos["local"].predict(X), 0.01, None)
    mu = np.clip(modelos["visitante"].predict(X), 0.01, None)
    x, y = goles_local.to_numpy(), goles_visitante.to_numpy()

    candidatos_rho = np.linspace(-0.3, 0.3, 121)
    log_verosimilitudes = [log_verosimilitud_dixon_coles(r, lam, mu, x, y) for r in candidatos_rho]
    return float(candidatos_rho[np.argmax(log_verosimilitudes)])


RHO_DIXON_COLES = ajustar_rho_dixon_coles(modelos_glm, X_train_inicial, df_train_inicial["goles_local"], df_train_inicial["goles_visitante"])
print(f"Corrección de Dixon-Coles: rho = {RHO_DIXON_COLES:.4f} (calibrado una vez sobre el entrenamiento pre-Mundial; "
      f"no se recalibra en el walk-forward, 13-32 partidos nuevos no lo van a mover sobre {len(df_train_inicial):,} históricos).")


def probabilidades_1x2_desde_lambdas(lam_local: np.ndarray, lam_visitante: np.ndarray,
                                       max_goles: int = 10) -> np.ndarray:
    '''Convierte vectores de (lambda_local, lambda_visitante) en probabilidades
    [P(local), P(empate), P(visitante)] recorriendo la rejilla conjunta de
    Poisson para cada partido, con la corrección de Dixon-Coles aplicada a los
    4 marcadores bajos -- la suma exacta de masa de probabilidad en cada
    región del marcador, no una aproximación cerrada.
    '''
    goles = np.arange(max_goles + 1)
    salida = np.zeros((len(lam_local), 3))
    for i, (la, lv) in enumerate(zip(lam_local, lam_visitante)):
        rejilla = np.outer(poisson.pmf(goles, la), poisson.pmf(goles, lv))
        rejilla[0, 0] *= 1 - la * lv * RHO_DIXON_COLES
        rejilla[0, 1] *= 1 + la * RHO_DIXON_COLES
        rejilla[1, 0] *= 1 + lv * RHO_DIXON_COLES
        rejilla[1, 1] *= 1 - RHO_DIXON_COLES
        rejilla = np.clip(rejilla, 0, None)
        salida[i] = [np.tril(rejilla, -1).sum(), np.trace(rejilla), np.triu(rejilla, 1).sum()]
    return salida / salida.sum(axis=1, keepdims=True)


def evaluar_modelo_goles(modelos: dict, X: pd.DataFrame, df_partidos: pd.DataFrame, nombre: str) -> dict:
    pred_local = np.clip(modelos["local"].predict(X), 0.01, None)
    pred_visitante = np.clip(modelos["visitante"].predict(X), 0.01, None)

    rmse = np.sqrt(mean_squared_error(
        pd.concat([df_partidos["goles_local"], df_partidos["goles_visitante"]]),
        np.concatenate([pred_local, pred_visitante]),
    ))

    probs = probabilidades_1x2_desde_lambdas(pred_local, pred_visitante)
    y_real = df_partidos["resultado_1x2"].map({"LOCAL": 0, "EMPATE": 1, "VISITANTE": 2}).values
    logloss = log_loss(y_real, probs, labels=[0, 1, 2])

    print(f"[{nombre}] RMSE goles: {rmse:.4f}  |  LogLoss 1X2: {logloss:.4f}")
    return {"rmse": rmse, "logloss": logloss}


print("--- Fase de grupos ya jugada ---")
metricas_glm = evaluar_modelo_goles(modelos_glm, X_test_grupos, df_test_grupos, "Poisson GLM")
metricas_lgbm_base = evaluar_modelo_goles(modelos_lgbm_base, X_test_grupos, df_test_grupos, "LightGBM (sin ajustar)")
metricas_lgbm_optuna = evaluar_modelo_goles(modelos_lgbm_optuna, X_test_grupos, df_test_grupos, "LightGBM (Optuna)")

candidatos = {"glm": (modelos_glm, metricas_glm), "lgbm_optuna": (modelos_lgbm_optuna, metricas_lgbm_optuna)}
nombre_ganador = min(candidatos, key=lambda k: candidatos[k][1]["rmse"])
modelos_actuales = candidatos[nombre_ganador][0]
print(f"\nModelo seleccionado para el walk-forward: {nombre_ganador}")

### Predicciones de fase de grupos, partido a partido

La evaluación de arriba solo guarda el RMSE/LogLoss **agregado** de los 72 partidos de
grupos — útil para elegir familia, pero no deja ver qué predijo el modelo en un partido
concreto. Aquí se guarda esa vista partido a partido, con el modelo ganador entrenado
solo con datos pre-Mundial (el mismo que ya se evaluó arriba "en frío"). A diferencia de
eliminatoria, en fase de grupos **no se predice "quién avanza"**: la clasificación de un
grupo depende de la tabla completa (puntos, diferencia de goles...), no del resultado de
un único partido, así que aquí solo tiene sentido predecir goles y 1X2.

In [ ]:
def predecir_partidos_grupos(modelos: dict, X: pd.DataFrame, df_partidos: pd.DataFrame) -> pd.DataFrame:
    '''Predicción partido a partido de la fase de grupos. Sin `resolver_eliminatoria`
    a propósito: ese concepto de "quién avanza" solo aplica a un cruce directo de
    eliminatoria, no a un partido de grupos.'''
    pred_local = np.clip(modelos["local"].predict(X), 0.01, None)
    pred_visitante = np.clip(modelos["visitante"].predict(X), 0.01, None)
    probs = probabilidades_1x2_desde_lambdas(pred_local, pred_visitante)
    etiquetas_1x2 = np.array(["LOCAL", "EMPATE", "VISITANTE"])

    filas = []
    for (_, fila), lam_l, lam_v, prob in zip(df_partidos.iterrows(), pred_local, pred_visitante, probs):
        filas.append({
            "fecha": fila["fecha"],
            "equipo_local": fila["equipo_local"],
            "equipo_visitante": fila["equipo_visitante"],
            "lambda_local": lam_l,
            "lambda_visitante": lam_v,
            # La moda de una Poisson(lambda) es siempre floor(lambda), nunca
            # round(lambda) -- para lambda=0.9, por ejemplo, 0 goles es más
            # probable que 1, aunque redondee a 1.
            "marcador_previsto": f"{int(np.floor(lam_l))}-{int(np.floor(lam_v))}",
            "prob_local": prob[0],
            "prob_empate": prob[1],
            "prob_visitante": prob[2],
            "resultado_1x2_previsto": etiquetas_1x2[np.argmax(prob)],
            "marcador_real": f"{int(fila['goles_local'])}-{int(fila['goles_visitante'])}",
            "resultado_1x2_real": fila["resultado_1x2"],
        })
    return pd.DataFrame(filas)


df_predicciones_grupos = predecir_partidos_grupos(candidatos[nombre_ganador][0], X_test_grupos, df_test_grupos)
ruta_predicciones_grupos = DIR_PROCESSED / "predicciones_fase_grupos.csv"
df_predicciones_grupos.to_csv(ruta_predicciones_grupos, index=False)

aciertos_1x2 = (df_predicciones_grupos["resultado_1x2_previsto"] == df_predicciones_grupos["resultado_1x2_real"]).sum()
aciertos_marcador = (df_predicciones_grupos["marcador_previsto"] == df_predicciones_grupos["marcador_real"]).sum()
print(f"Guardado: {ruta_predicciones_grupos} ({len(df_predicciones_grupos)} partidos)")
print(f"Acierto 1X2: {aciertos_1x2}/{len(df_predicciones_grupos)}  |  Acierto marcador exacto: {aciertos_marcador}/{len(df_predicciones_grupos)}")
df_predicciones_grupos.head()

### Checkpoint: guardar un snapshot del modelo por etapa, no solo el último

Hasta ahora `models/` solo guardaba el estado **más reciente**, sobreescrito cada vez
— no había forma de recuperar qué creía el modelo antes de la fase de grupos, o antes
de dieciseisavos, una vez reentrenado con la ronda siguiente. `guardar_checkpoint`
escribe un snapshot con nombre (`models/checkpoints/<etapa>/`) además de refrescar
`models/` con el estado actual, para poder comparar más adelante "qué predecía el
modelo en cada momento del torneo" — y para que el Notebook 6 pueda reutilizar el
`pre_mundial` en vez de reentrenarlo desde cero.

In [7]:
def guardar_checkpoint(modelos: dict, nombre_etapa: str, familia: str, historico: pd.DataFrame,
                        metadatos_extra: dict | None = None) -> None:
    """Persiste el modelo dos veces: en `models/checkpoints/<nombre_etapa>/`
    (snapshot fijo de esa etapa, no se vuelve a tocar) y en `models/` (el
    estado "actual", que cada llamada posterior sí sobreescribe)."""
    metadatos = {
        "etapa": nombre_etapa,
        "familia": familia,
        "features": FEATURES_MODELO,
        "fecha_entrenamiento_utc": datetime.now(timezone.utc).isoformat(),
        "n_partidos_entrenamiento": len(historico),
        "ultima_fecha_entrenamiento": str(historico["fecha"].max().date()),
        **(metadatos_extra or {}),
    }
    for destino in (DIR_MODELOS / "checkpoints" / nombre_etapa, DIR_MODELOS):
        destino.mkdir(parents=True, exist_ok=True)
        joblib.dump(modelos["local"], destino / "modelo_goles_local.joblib")
        joblib.dump(modelos["visitante"], destino / "modelo_goles_visitante.joblib")
        with open(destino / "metadata.json", "w") as f:
            json.dump(metadatos, f, indent=2, default=str)
    print(f"Checkpoint '{nombre_etapa}' guardado ({len(historico):,} partidos, hasta "
          f"{metadatos['ultima_fecha_entrenamiento']}).")


guardar_checkpoint(
    candidatos[nombre_ganador][0], "pre_mundial", nombre_ganador, df_train_inicial,
    {"metricas_fase_grupos": {"glm": metricas_glm, "lgbm_base": metricas_lgbm_base, "lgbm_optuna": metricas_lgbm_optuna}},
)

Checkpoint 'pre_mundial' guardado (32,286 partidos, hasta 2026-06-10).


## 4.6 Matriz de probabilidad conjunta y resolución de eliminatoria

Quién avanza se decide comparando **siempre** la probabilidad acumulada de cada lado en
la matriz de probabilidad conjunta (recorriendo todos los marcadores posibles de 0 a
`MAX_GOLES_MATRIZ`) — nunca comparando los marcadores más probables de cada lambda por
separado. Esa comparación puntual es más frágil de lo que parece: la moda de una
Poisson(λ) por sí sola no es lo mismo que "quién tiene más probabilidad de ganar el
partido", así que apoyarse en ella (con `round()` o incluso con `floor()`, que sí es la
moda correcta) para decidir el ganador seguiría siendo una aproximación innecesaria
cuando ya se dispone de la propia matriz de probabilidad.

In [ ]:
def matriz_probabilidad_conjunta(lam_a: float, lam_b: float, max_goles: int = MAX_GOLES_MATRIZ) -> pd.DataFrame:
    '''Rejilla P(goles_A=i, goles_B=j) = Poisson(i;lam_a) * Poisson(j;lam_b),
    con la corrección de Dixon-Coles en los 4 marcadores bajos (mismo rho que
    en 4.5, calibrado una sola vez sobre el entrenamiento pre-Mundial).'''
    goles = np.arange(max_goles + 1)
    matriz = np.outer(poisson.pmf(goles, lam_a), poisson.pmf(goles, lam_b))
    matriz[0, 0] *= 1 - lam_a * lam_b * RHO_DIXON_COLES
    matriz[0, 1] *= 1 + lam_a * RHO_DIXON_COLES
    matriz[1, 0] *= 1 + lam_b * RHO_DIXON_COLES
    matriz[1, 1] *= 1 - RHO_DIXON_COLES
    matriz = np.clip(matriz, 0, None)
    matriz = matriz / matriz.sum()  # la corrección no preserva exactamente la masa total
    return pd.DataFrame(matriz, index=[f"A={i}" for i in goles], columns=[f"B={j}" for j in goles])


def resolver_eliminatoria(equipo_a: str, equipo_b: str, lam_a: float, lam_b: float) -> dict:
    '''Decide quién avanza comparando SIEMPRE la probabilidad acumulada de cada
    lado en la matriz de probabilidad conjunta -- nunca un marcador puntual de
    cada lambda por separado (ver 4.6). `marcador_previsto` se deja solo a
    título informativo: es la moda de cada Poisson (floor(lambda)), no lo que
    decide quién gana.
    '''
    matriz = matriz_probabilidad_conjunta(lam_a, lam_b).values
    prob_gana_a = np.tril(matriz, -1).sum()
    prob_gana_b = np.triu(matriz, 1).sum()
    ganador = equipo_a if prob_gana_a > prob_gana_b else equipo_b

    return {"equipo_a": equipo_a, "equipo_b": equipo_b,
            "marcador_previsto": f"{int(np.floor(lam_a))}-{int(np.floor(lam_b))}",
            "ganador": ganador, "metodo": f"matriz_conjunta (P(A)={prob_gana_a:.3f} vs P(B)={prob_gana_b:.3f})"}


# Ejemplo ilustrativo: un cruce parejo.
ejemplo = resolver_eliminatoria("Equipo A", "Equipo B", lam_a=1.35, lam_b=1.28)
print(ejemplo)
matriz_probabilidad_conjunta(1.35, 1.28).round(3)

## 4.7 Bucle de walk-forward: eliminatorias con reentrenamiento continuo

Las rondas eliminatorias se agrupan por fecha (un salto de ≥2 días sin partidos
señala el cambio de ronda — no hace falta ninguna estructura de bracket externa,
solo el propio calendario real de partidos). Para cada ronda:

1. Se evalúa con el modelo tal como quedó **entrenado hasta ANTES** de esa ronda
   (nunca con datos que ya la incluyan — la misma disciplina de holdout de todo el
   proyecto).
2. Cada partido empatado a 90' se resuelve con la matriz de probabilidad conjunta.
3. Solo DESPUÉS de evaluar, los resultados reales de esa ronda se incorporan al
   histórico de entrenamiento — reentrenamiento completo (no incremental: con este
   volumen de datos el ajuste tarda segundos, y evita mantener a mano el estado
   incremental de las ventanas móviles) — antes de pasar a la siguiente ronda.

In [9]:
def agrupar_en_rondas(df_eliminatoria: pd.DataFrame, salto_dias: int = 2) -> list[pd.DataFrame]:
    '''Agrupa partidos de eliminatoria en rondas por proximidad de fecha.'''
    jugados = df_eliminatoria[df_eliminatoria["jugado"]].sort_values("fecha")
    if jugados.empty:
        return []
    salto = jugados["fecha"].diff().dt.days.fillna(0) > salto_dias
    jugados = jugados.assign(_ronda=salto.cumsum())
    return [grupo for _, grupo in jugados.groupby("_ronda")]


def reentrenar_modelos(df_historico_actualizado: pd.DataFrame, familia: str) -> dict:
    '''Reentrena desde cero (no incremental) la familia de modelo indicada
    ("glm" o "lgbm_optuna") con TODO el histórico ya actualizado.
    '''
    X = df_historico_actualizado[FEATURES_MODELO]
    if familia == "glm":
        return {
            "local": entrenar_poisson_glm(X, df_historico_actualizado["goles_local"]),
            "visitante": entrenar_poisson_glm(X, df_historico_actualizado["goles_visitante"]),
        }
    return {
        "local": entrenar_lightgbm_poisson(X, df_historico_actualizado["goles_local"], estudio_local.best_params),
        "visitante": entrenar_lightgbm_poisson(X, df_historico_actualizado["goles_visitante"], estudio_visitante.best_params),
    }


rondas = agrupar_en_rondas(df_eliminatoria)
print(f"Rondas de eliminatoria detectadas en el histórico (ya jugadas): {len(rondas)}")
for i, ronda in enumerate(rondas, start=1):
    print(f"  Ronda {i}: {ronda['fecha'].min().date()} -> {ronda['fecha'].max().date()}, {len(ronda)} partidos")

Rondas de eliminatoria detectadas en el histórico (ya jugadas): 1
  Ronda 1: 2026-06-28 -> 2026-07-02, 13 partidos


In [10]:
def probabilidades_partido(lam_local: float, lam_visitante: float) -> tuple[float, float, float]:
    '''Wrapper de una sola fila sobre `probabilidades_1x2_desde_lambdas`, para
    no repetir la lógica de la rejilla al construir el registro de cada
    predicción dentro del bucle.'''
    probs = probabilidades_1x2_desde_lambdas(np.array([lam_local]), np.array([lam_visitante]))
    return tuple(probs[0])


# Nombre legible de la etapa según el nº de partidos de la ronda (bracket de 32
# equipos: dieciseisavos=16, octavos=8, cuartos=4, semis=2, final=1); si no
# encaja (p.ej. una ronda incompleta, como ahora mismo) se usa "ronda_i".
ETAPAS_POR_TAMANO = {16: "dieciseisavos", 8: "octavos", 4: "cuartos", 2: "semis", 1: "final"}

historico_acumulado = pd.concat([df_train_inicial, df_test_grupos], ignore_index=True)
modelos_actuales = reentrenar_modelos(historico_acumulado, nombre_ganador)
print(f"Fase de grupos incorporada al entrenamiento ({len(historico_acumulado):,} partidos acumulados).")
guardar_checkpoint(modelos_actuales, "post_grupos", nombre_ganador, historico_acumulado)

resumen_walkforward = []
predicciones_partidos = []  # una fila por partido: predicción + resultado real, para poder auditar el modelo partido a partido

for i, ronda in enumerate(rondas, start=1):
    print(f"\n=== Ronda {i} ({len(ronda)} partidos, {ronda['fecha'].min().date()}) ===")

    # 1. Evaluar ESTA ronda con el modelo tal como estaba ANTES de conocerla.
    X_ronda = ronda[FEATURES_MODELO]
    metricas_ronda = evaluar_modelo_goles(modelos_actuales, X_ronda, ronda, f"ronda {i} (modelo pre-ronda)")

    # 2. Resolver cada partido, con matriz conjunta si el marcador previsto es empate.
    pred_local = np.clip(modelos_actuales["local"].predict(X_ronda), 0.01, None)
    pred_visitante = np.clip(modelos_actuales["visitante"].predict(X_ronda), 0.01, None)
    for (_, fila), lam_l, lam_v in zip(ronda.iterrows(), pred_local, pred_visitante):
        resultado = resolver_eliminatoria(fila["equipo_local"], fila["equipo_visitante"], lam_l, lam_v)
        prob_local, prob_empate, prob_visitante = probabilidades_partido(lam_l, lam_v)
        print(f"  {resultado['equipo_a']} vs {resultado['equipo_b']}: "
              f"previsto {resultado['marcador_previsto']}, avanza {resultado['ganador']} ({resultado['metodo']})")
        # Nota: "marcador_real_90min" es el resultado reglamentario (lo único que
        # modela el sistema); en un cruce que se decidiera en prórroga/penaltis
        # esto NO tiene por qué coincidir con quién avanzó realmente al torneo.
        predicciones_partidos.append({
            "ronda": i,
            "fecha": fila["fecha"],
            "equipo_local": fila["equipo_local"],
            "equipo_visitante": fila["equipo_visitante"],
            "lambda_local": lam_l,
            "lambda_visitante": lam_v,
            "marcador_previsto": resultado["marcador_previsto"],
            "prob_local": prob_local,
            "prob_empate": prob_empate,
            "prob_visitante": prob_visitante,
            "avanza_previsto": resultado["ganador"],
            "metodo_desempate": resultado["metodo"],
            "marcador_real_90min": f"{int(fila['goles_local'])}-{int(fila['goles_visitante'])}",
            "resultado_1x2_real": fila["resultado_1x2"],
        })

    resumen_walkforward.append({"ronda": i, "n_partidos": len(ronda), **metricas_ronda})

    # 3. Solo AHORA se incorporan los resultados reales de esta ronda, y se
    #    reentrena de cara a la siguiente — el "rolling update" del pipeline.
    historico_acumulado = pd.concat([historico_acumulado, ronda], ignore_index=True)
    modelos_actuales = reentrenar_modelos(historico_acumulado, nombre_ganador)
    nombre_etapa = f"post_{ETAPAS_POR_TAMANO.get(len(ronda), f'ronda_{i}')}"
    guardar_checkpoint(modelos_actuales, nombre_etapa, nombre_ganador, historico_acumulado, {"metricas_ronda": metricas_ronda})
    print(f"  Reentrenado con {len(historico_acumulado):,} partidos acumulados.")

if resumen_walkforward:
    print("\n=== Evolución de las métricas ronda a ronda ===")
    print(pd.DataFrame(resumen_walkforward).round(4).to_string(index=False))
else:
    print("Sin rondas de eliminatoria jugadas todavía en el histórico disponible.")

Fase de grupos incorporada al entrenamiento (32,358 partidos acumulados).


Checkpoint 'post_grupos' guardado (32,358 partidos, hasta 2026-06-27).

=== Ronda 1 (13 partidos, 2026-06-28) ===
[ronda 1 (modelo pre-ronda)] RMSE goles: 0.6877  |  LogLoss 1X2: 0.8199
  South Africa vs Canada: previsto 1-1, avanza Canada (matriz_conjunta (P(A)=0.300 vs P(B)=0.432))
  Germany vs Paraguay: previsto 3-1, avanza Germany (marcador_90min)
  Brazil vs Japan: previsto 2-1, avanza Brazil (marcador_90min)
  Netherlands vs Morocco: previsto 2-1, avanza Netherlands (marcador_90min)
  France vs Sweden: previsto 3-1, avanza France (marcador_90min)
  Ivory Coast vs Norway: previsto 1-1, avanza Ivory Coast (matriz_conjunta (P(A)=0.402 vs P(B)=0.325))
  Mexico vs Ecuador: previsto 2-1, avanza Mexico (marcador_90min)
  England vs DR Congo: previsto 3-1, avanza England (marcador_90min)
  United States vs Bosnia and Herzegovina: previsto 2-1, avanza United States (marcador_90min)
  Belgium vs Senegal: previsto 2-1, avanza Belgium (marcador_90min)
  Spain vs Austria: previsto 2-1, avanza

Checkpoint 'post_ronda_1' guardado (32,371 partidos, hasta 2026-07-02).
  Reentrenado con 32,371 partidos acumulados.

=== Evolución de las métricas ronda a ronda ===
 ronda  n_partidos   rmse  logloss
     1          13 0.6877   0.8199


## 4.8 Persistencia: modelo final y predicciones

El resto del notebook vivía solo en memoria de la sesión — ni el modelo ganador
ni las predicciones partido a partido quedaban en disco, así que reabrir el
notebook obligaba a reentrenar todo desde cero para volver a consultarlas. Dos
artefactos, ambos derivados y por tanto en `.gitignore`:

- `data/processed/predicciones_eliminatoria.csv`: una fila por partido de
  eliminatoria con la predicción completa (lambdas, marcador previsto,
  probabilidades 1X2, método de desempate) y el resultado real al lado, para
  poder auditar el modelo partido a partido sin releer los prints del bucle.
- `models/`: el modelo ganador (`nombre_ganador`), reentrenado con **todo** el
  histórico acumulado hasta la última ronda conocida — es el que se usaría
  para predecir la siguiente ronda todavía no jugada — junto a un
  `metadata.json` con de qué familia es, qué features espera, cuándo se
  entrenó y con qué métricas.


In [11]:
DIR_PREDICCIONES = DIR_PROCESSED  # mismo directorio que el resto de artefactos derivados del pipeline

df_predicciones = pd.DataFrame(predicciones_partidos)
ruta_predicciones = DIR_PREDICCIONES / "predicciones_eliminatoria.csv"
df_predicciones.to_csv(ruta_predicciones, index=False)
print(f"Predicciones de eliminatoria guardadas: {ruta_predicciones} ({len(df_predicciones)} partidos)")
df_predicciones


Predicciones de eliminatoria guardadas: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/data/processed/predicciones_eliminatoria.csv (13 partidos)


,ronda,fecha,equipo_local,equipo_visitante,lambda_local,lambda_visitante,marcador_previsto,prob_local,prob_empate,prob_visitante,avanza_previsto,metodo_desempate,marcador_real_90min,resultado_1x2_real
0,1,2026-06-28,South Africa,Canada,1.142594,1.428974,1-1,0.301642,0.262357,0.436002,Canada,matriz_conjunta (P(A)=0.300 vs P(B)=0.432),0-1,VISITANTE
1,1,2026-06-29,Germany,Paraguay,2.548989,0.630273,3-1,0.788924,0.140646,0.070430,Germany,marcador_90min,1-1,EMPATE
2,1,2026-06-29,Brazil,Japan,1.657996,0.985452,2-1,0.531317,0.244328,0.224355,Brazil,marcador_90min,2-1,LOCAL
3,1,2026-06-29,Netherlands,Morocco,1.607628,1.012542,2-1,0.512300,0.249166,0.238534,Netherlands,marcador_90min,1-1,EMPATE
4,1,2026-06-30,France,Sweden,2.843761,0.550736,3-1,0.842178,0.110667,0.047155,France,marcador_90min,3-0,LOCAL
5,1,2026-06-30,Ivory Coast,Norway,1.329266,1.164650,1-1,0.404570,0.269312,0.326118,Ivory Coast,matriz_conjunta (P(A)=0.402 vs P(B)=0.325),1-2,VISITANTE
6,1,2026-06-30,Mexico,Ecuador,1.945078,0.816929,2-1,0.638786,0.211191,0.150023,Mexico,marcador_90min,2-0,LOCAL
7,1,2026-07-01,England,DR Congo,2.682122,0.601883,3-1,0.812764,0.127262,0.059974,England,marcador_90min,2-1,LOCAL
8,1,2026-07-01,United States,Bosnia and Herzegovina,1.966741,0.809803,2-1,0.645113,0.208610,0.146277,United States,marcador_90min,2-0,LOCAL
9,1,2026-07-01,Belgium,Senegal,2.153105,0.742877,2-1,0.698386,0.185895,0.115719,Belgium,marcador_90min,2-2,EMPATE


In [12]:
import json
from datetime import datetime, timezone

import joblib

DIR_MODELOS = DIR_RAIZ / "models"
DIR_MODELOS.mkdir(exist_ok=True)

joblib.dump(modelos_actuales["local"], DIR_MODELOS / "modelo_goles_local.joblib")
joblib.dump(modelos_actuales["visitante"], DIR_MODELOS / "modelo_goles_visitante.joblib")

metadatos_modelo = {
    "familia": nombre_ganador,
    "features": FEATURES_MODELO,
    "fecha_entrenamiento_utc": datetime.now(timezone.utc).isoformat(),
    "n_partidos_entrenamiento": len(historico_acumulado),
    "ultima_fecha_entrenamiento": str(historico_acumulado["fecha"].max().date()),
    "metricas_fase_grupos": {
        "glm": metricas_glm,
        "lgbm_base": metricas_lgbm_base,
        "lgbm_optuna": metricas_lgbm_optuna,
    },
    "metricas_walkforward_por_ronda": resumen_walkforward,
    "hiperparametros_lgbm_optuna": {
        "local": estudio_local.best_params,
        "visitante": estudio_visitante.best_params,
    } if nombre_ganador == "lgbm_optuna" else None,
}
with open(DIR_MODELOS / "metadata.json", "w") as f:
    json.dump(metadatos_modelo, f, indent=2, default=str)

print(f"Modelo '{nombre_ganador}' persistido en {DIR_MODELOS} (entrenado con {len(historico_acumulado):,} partidos, "
      f"hasta {metadatos_modelo['ultima_fecha_entrenamiento']}).")


Modelo 'glm' persistido en /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/models (entrenado con 32,371 partidos, hasta 2026-07-02).


## 4.9 Predicción de los próximos partidos (todavía sin jugar)

Todo lo anterior evalúa partidos que **ya se jugaron** (para poder comparar contra el
resultado real). Esta sección hace lo que de verdad importa de cara a lo que queda de
torneo: usar el modelo actual (`modelos_actuales`, ya reentrenado con todo lo conocido
hasta la última ronda) para predecir los cruces de eliminatoria **pendientes** —
incluidos los de octavos que ya se conocen aunque todavía no se hayan disputado
(algunos octavos siguen bloqueados por un dieciseisavo de hoy sin resolver, p. ej.
`"W83"`/`"W88"` en el calendario original; esas filas ni siquiera llegan a
`partidos_features.csv` hasta que el cruce anterior se juega, así que se filtran por
seguridad si alguna vez aparecieran).

In [13]:
partidos_pendientes = df_eliminatoria[~df_eliminatoria["jugado"]].copy()

# Defensivo: un cruce de bracket sin resolver todavía (p.ej. "W83") no tiene
# historial de Elo/forma propio -- no hay nada que predecir para él.
patron_placeholder = r"^[WL]\d+$"
partidos_pendientes = partidos_pendientes[
    ~partidos_pendientes["equipo_local"].str.match(patron_placeholder)
    & ~partidos_pendientes["equipo_visitante"].str.match(patron_placeholder)
].sort_values("fecha")

predicciones_pendientes = []
if not partidos_pendientes.empty:
    X_pendientes = partidos_pendientes[FEATURES_MODELO]
    pred_local = np.clip(modelos_actuales["local"].predict(X_pendientes), 0.01, None)
    pred_visitante = np.clip(modelos_actuales["visitante"].predict(X_pendientes), 0.01, None)

    print(f"=== Predicción con el modelo actual ({nombre_ganador}, entrenado hasta "
          f"{historico_acumulado['fecha'].max().date()}) ===")
    for (_, fila), lam_l, lam_v in zip(partidos_pendientes.iterrows(), pred_local, pred_visitante):
        resultado = resolver_eliminatoria(fila["equipo_local"], fila["equipo_visitante"], lam_l, lam_v)
        prob_local, prob_empate, prob_visitante = probabilidades_partido(lam_l, lam_v)
        print(f"  [{fila['fecha'].date()}] {resultado['equipo_a']} vs {resultado['equipo_b']}: "
              f"previsto {resultado['marcador_previsto']}, avanza {resultado['ganador']} ({resultado['metodo']})")
        predicciones_pendientes.append({
            "fecha": fila["fecha"],
            "equipo_local": fila["equipo_local"],
            "equipo_visitante": fila["equipo_visitante"],
            "lambda_local": lam_l,
            "lambda_visitante": lam_v,
            "marcador_previsto": resultado["marcador_previsto"],
            "prob_local": prob_local,
            "prob_empate": prob_empate,
            "prob_visitante": prob_visitante,
            "avanza_previsto": resultado["ganador"],
            "metodo_desempate": resultado["metodo"],
        })
else:
    print("No hay partidos de eliminatoria pendientes con ambos equipos ya conocidos.")

df_predicciones_pendientes = pd.DataFrame(predicciones_pendientes)
ruta_predicciones_pendientes = DIR_PROCESSED / "predicciones_proximos_partidos.csv"
df_predicciones_pendientes.to_csv(ruta_predicciones_pendientes, index=False)
print(f"\nGuardado: {ruta_predicciones_pendientes} ({len(df_predicciones_pendientes)} partidos)")
df_predicciones_pendientes

=== Predicción con el modelo actual (glm, entrenado hasta 2026-07-02) ===
  [2026-07-03] Argentina vs Cape Verde: previsto 4-0, avanza Argentina (marcador_90min)
  [2026-07-03] Colombia vs Ghana: previsto 3-1, avanza Colombia (marcador_90min)
  [2026-07-03] Australia vs Egypt: previsto 1-1, avanza Australia (matriz_conjunta (P(A)=0.416 vs P(B)=0.316))
  [2026-07-04] Paraguay vs France: previsto 1-2, avanza France (marcador_90min)
  [2026-07-04] Canada vs Morocco: previsto 1-1, avanza Canada (matriz_conjunta (P(A)=0.382 vs P(B)=0.349))
  [2026-07-05] Mexico vs England: previsto 2-1, avanza Mexico (marcador_90min)
  [2026-07-05] Brazil vs Norway: previsto 2-1, avanza Brazil (marcador_90min)
  [2026-07-06] Portugal vs Spain: previsto 1-1, avanza Portugal (matriz_conjunta (P(A)=0.372 vs P(B)=0.357))
  [2026-07-06] United States vs Belgium: previsto 1-1, avanza United States (matriz_conjunta (P(A)=0.370 vs P(B)=0.358))

Guardado: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/d

,fecha,equipo_local,equipo_visitante,lambda_local,lambda_visitante,marcador_previsto,prob_local,prob_empate,prob_visitante,avanza_previsto,metodo_desempate
0,2026-07-03,Argentina,Cape Verde,3.795599,0.415331,4-0,0.935805,0.049308,0.014887,Argentina,marcador_90min
1,2026-07-03,Colombia,Ghana,3.075972,0.520119,3-1,0.871266,0.092356,0.036378,Colombia,marcador_90min
2,2026-07-03,Australia,Egypt,1.388429,1.172398,1-1,0.418647,0.264399,0.316954,Australia,matriz_conjunta (P(A)=0.416 vs P(B)=0.316)
3,2026-07-04,Paraguay,France,0.824749,2.026719,1-2,0.143266,0.202985,0.653749,France,marcador_90min
4,2026-07-04,Canada,Morocco,1.334097,1.262597,1-1,0.384802,0.263909,0.351289,Canada,matriz_conjunta (P(A)=0.382 vs P(B)=0.349)
5,2026-07-05,Mexico,England,1.515267,1.052580,2-1,0.479307,0.257375,0.263317,Mexico,marcador_90min
6,2026-07-05,Brazil,Norway,1.801100,0.898560,2-1,0.586681,0.228823,0.184496,Brazil,marcador_90min
7,2026-07-06,Portugal,Spain,1.298559,1.266179,1-1,0.374638,0.265987,0.359376,Portugal,matriz_conjunta (P(A)=0.372 vs P(B)=0.357)
8,2026-07-06,United States,Belgium,1.280845,1.256357,1-1,0.371953,0.267692,0.360356,United States,matriz_conjunta (P(A)=0.370 vs P(B)=0.358)


## Resumen

- Modelo de goles seleccionado (RMSE + LogLoss en fase de grupos): impreso en la
  sección 4.5 (`nombre_ganador`).
- El bucle de la sección 4.7 demuestra el ciclo completo de un sistema de predicción
  vivo: evaluar honestamente ANTES de conocer el resultado, e incorporar el dato real
  DESPUÉS — nunca al revés.
- La matriz de probabilidad conjunta (4.6) es la pieza que resuelve la ambigüedad de
  un empate a 90 minutos en una eliminatoria directa, sin necesitar simular
  prórroga/penaltis explícitamente: el equipo con más probabilidad acumulada en su
  mitad del marcador es el que se da por ganador.